# Pars form bin log file

## 1. Probedaten einlesen

In [65]:
from pathlib import Path

# Pfad festlegen
project_path = Path.cwd().parent
folder = 'example_output'
filename = 'bin_24102025_1434'
filepath = project_path / folder / filename

# Datei im BINARY-Modus ('rb') einlesen
try:
    with open(filepath, 'rb') as f:         # WICHTIG: 'rb' für 'read binary' verwenden
        binary_data = f.read()
except Exception as e:
    print(f"Fehler beim Lesen der Datei: {e}")

# Überprüfen des Datentyps
print(f"Bytes: {binary_data[:50]}")
print(type(binary_data))
print(type(binary_data[0]))

Bytes: b'0 0 0 0 0 0 0 0 0 1 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 '
<class 'bytes'>
<class 'int'>


## 2.1 Umwandeln in bin list

In [66]:
# Jedes Byte in einen 8-stelligen Binär-String umwandeln
byte_list = [f'{byte:08b}' for byte in binary_data]

print(f"Byte list: {byte_list[:10]}")
print(type(byte_list))
print(type(byte_list[0]))

Byte list: ['00110000', '00100000', '00110000', '00100000', '00110000', '00100000', '00110000', '00100000', '00110000', '00100000']
<class 'list'>
<class 'str'>


## 2.2 Umwandeln in int list

In [67]:
integer_liste = list(binary_data)

print(f"Int list: {integer_liste[:30]}")
print(type(integer_liste))
print(type(integer_liste[0]))

Int list: [48, 32, 48, 32, 48, 32, 48, 32, 48, 32, 48, 32, 48, 32, 48, 32, 48, 32, 49, 32, 49, 32, 48, 32, 48, 32, 49, 32, 48, 32]
<class 'list'>
<class 'int'>


## 2.3 Umwandeln in hex list

In [68]:
hex_liste = [format(byte, '02x') for byte in binary_data]

print(f"Hex list: {hex_liste[:30]}")   
print(type(hex_liste))
print(type(hex_liste[0])) 

Hex list: ['30', '20', '30', '20', '30', '20', '30', '20', '30', '20', '30', '20', '30', '20', '30', '20', '30', '20', '31', '20', '31', '20', '30', '20', '30', '20', '31', '20', '30', '20']
<class 'list'>
<class 'str'>


## 3. Vergleiche 

In [69]:
print(f"Bytes: {binary_data[:50]}")
print(f"Byte list: {byte_list[:10]}")
print(f"Int list: {integer_liste[:30]}")
print(f"Hex list: {hex_liste[:30]}")

Bytes: b'0 0 0 0 0 0 0 0 0 1 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0 '
Byte list: ['00110000', '00100000', '00110000', '00100000', '00110000', '00100000', '00110000', '00100000', '00110000', '00100000']
Int list: [48, 32, 48, 32, 48, 32, 48, 32, 48, 32, 48, 32, 48, 32, 48, 32, 48, 32, 49, 32, 49, 32, 48, 32, 48, 32, 49, 32, 48, 32]
Hex list: ['30', '20', '30', '20', '30', '20', '30', '20', '30', '20', '30', '20', '30', '20', '30', '20', '30', '20', '31', '20', '31', '20', '30', '20', '30', '20', '31', '20', '30', '20']


## 3. Header parsen

Nachdem die Daten korrekt als `bytes` eingelesen wurden, können wir den Header parsen. Der folgende Code extrahiert die ersten 14 Bytes und interpretiert sie gemäß der Spezifikation.

In [ ]:
def parse_header_from_int_list(int_list):
    """
    Parst den Header aus einer Liste von Integer-Werten.
    """
    if len(int_list) < 14:
        print("Fehler: Nicht genügend Daten (Integer-Werte) für den Header vorhanden.")
        return None
    try:
        # --- Magic (8 bytes) ---
        # Nimmt die ersten 8 Integer und wandelt sie in Zeichen um
        print(f"Test Int list: {int_list[:30]}")

        magic_chars = [chr(i) for i in int_list[0:8]]
        magic_string = "".join(magic_chars)

        # --- Packet ID (2 bytes) ---
        # Nimmt die nächsten 2 Integer und wandelt sie in Zeichen um
        packetid_chars = [chr(i) for i in int_list[8:10]]
        packetid_string = "".join(packetid_chars)

        # --- Length (4 bytes) ---
        # Nimmt die nächsten 4 Integer und fasst sie zu einer 32-bit-Ganzzahl zusammen
        length_bytes_as_ints = int_list[10:14]
        length_bytes = bytes(length_bytes_as_ints)
        length_uint32 = int.from_bytes(length_bytes, byteorder='little')

        return {"magic": magic_string, "packet_id": packetid_string, "length": length_uint32}    

    except (ValueError, TypeError) as e:
        print(f"Ein Fehler ist bei der Umwandlung aufgetreten: {e}")
        print("Stellen Sie sicher, dass die 'Int list' nur gültige Byte-Werte (0-255)enthält.")
        return None

# Rufen Sie die neue Funktion auf
header_from_int_list = parse_header_from_int_list(integer_liste)

if header_from_int_list:
    print("--- Analysierte Header-Informationen aus der Int list ---")
    print(f"Magic:     '{header_from_int_list['magic']}' (Erwartet: 'ECHOLOGG')")
    print(f"Packet ID: '{header_from_int_list['packet_id']}' (Erwartet: 'EC')")
    print(f"Length:    {header_from_int_list['length']}")

Test Int list: [48, 32, 48, 32, 48, 32, 48, 32, 48, 32, 48, 32, 48, 32, 48, 32, 48, 32, 49, 32, 49, 32, 48, 32, 48, 32, 49, 32, 48, 32]
--- Analysierte Header-Informationen aus der Int list ---
Magic:     '0 0 0 0 ' (Erwartet: 'ECHOLOGG')
Packet ID: '0 ' (Erwartet: 'EC')
Length:    540024880
